In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
N = 1000

# --- Segments and categories ---
segments    = np.random.choice(['Retail', 'Corporate', 'SME', 'Private Banking'], N, p=[0.4, 0.25, 0.25, 0.10])
regions     = np.random.choice(['Copenhagen', 'Aarhus', 'Odense', 'Aalborg'], N, p=[0.45, 0.25, 0.20, 0.10])
products    = np.random.choice(['Mortgage', 'Consumer Loan', 'Credit Card', 'Overdraft'], N, p=[0.35, 0.30, 0.20, 0.15])
collateral  = np.random.choice(['Real Estate', 'Financial Asset', 'Unsecured'], N, p=[0.45, 0.30, 0.25])
ratings     = np.random.choice(['AAA', 'AA', 'A', 'BBB', 'BB', 'B', 'CCC'], N, p=[0.10, 0.15, 0.20, 0.25, 0.15, 0.10, 0.05])

# --- Numeric features ---
income        = np.random.lognormal(mean=12.5, sigma=0.6, size=N).clip(20_000, 500_000)
loan_amount   = np.random.lognormal(mean=13.0, sigma=0.7, size=N).clip(10_000, 2_000_000)
LTV           = np.random.beta(2, 3, N) * 1.4
age           = np.random.randint(22, 75, N)
months_active = np.random.exponential(36, N).clip(1, 240).astype(int)

# --- Target: revenue with segment-level heteroskedasticity ---
base_revenue = (
    0.02 * loan_amount
    + 0.005 * income
    + np.where(segments == 'Private Banking', 8000, 0)
    + np.where(segments == 'Corporate',       4000, 0)
)
noise        = np.random.normal(0, 500 + 0.01 * loan_amount, N)
revenue      = (base_revenue + noise).clip(0)

# --- Default flag (imbalanced, ~12%) ---
default_prob = (
    0.05
    + 0.15 * LTV
    - 0.00001 * income
    + np.where(collateral == 'Unsecured', 0.08, 0)
    + np.where(ratings.isin(['B', 'CCC']) if hasattr(ratings, 'isin') else np.isin(ratings, ['B', 'CCC']), 0.12, 0)
).clip(0, 1)
default      = np.random.binomial(1, default_prob, N)

# --- Missing values (realistic: ~5-8% per column) ---
for col_arr, pct in [(income, 0.05), (LTV, 0.07), (collateral, 0.06)]:
    mask = np.random.rand(N) < pct
    if col_arr.dtype == object if hasattr(col_arr, 'dtype') else False:
        col_arr[mask] = None

# --- Assemble DataFrame ---
df = pd.DataFrame({
    'customer_id':    [f'CUS{str(i).zfill(5)}' for i in range(N)],
    'segment':        segments,
    'region':         regions,
    'product':        products,
    'collateral':     collateral,
    'rating':         ratings,
    'age':            age,
    'income':         income,
    'loan_amount':    loan_amount,
    'LTV':            LTV,
    'months_active':  months_active,
    'revenue':        revenue.round(2),
    'default':        default,
})

# Inject missings into the assembled DataFrame
for col, pct in [('income', 0.05), ('LTV', 0.07), ('collateral', 0.06), ('rating', 0.04)]:
    df.loc[df.sample(frac=pct, random_state=1).index, col] = np.nan

# --- Optional: save to CSV ---
df.to_csv('oneliners_2_data.csv', index=False)

print(f"Shape      : {df.shape}")
print(f"Default %  : {df['default'].mean()*100:.1f}%")
print(f"Missing %  :\n{df.isnull().mean().mul(100).round(1)[df.isnull().any()]}")
print(df.head(3).to_string())

In [ ]:
# Full missing value report as percentage
df.isnull().mean().mul(100).round(2).sort_values(ascending=False)

# Correlation of all numeric columns with a single target
df.corr()['LGD'].drop('LGD').sort_values(key=abs, ascending=False)

# Value counts as % with cumulative share — useful for distribution checks
df['rating'].value_counts(normalize=True).mul(100).cumsum().round(2)

# IQR-based outlier flag in one line
df['is_outlier'] = df['income'].apply(lambda x: x < df['income'].quantile(0.25) - 1.5 * (df['income'].quantile(0.75) - df['income'].quantile(0.25)) or x > df['income'].quantile(0.75) + 1.5 * (df['income'].quantile(0.75) - df['income'].quantile(0.25)))

# Group stats table — mean, std, count, median in one shot
df.groupby('segment')['revenue'].agg(['mean', 'std', 'count', 'median']).round(2)

# Flatten a MultiIndex after groupby
df.groupby(['region', 'product'])['sales'].sum().reset_index().rename(columns={'sales': 'total_sales'})

# Clip extreme values to 1st-99th percentile (Winsorisation)
df['LTV'] = df['LTV'].clip(*df['LTV'].quantile([0.01, 0.99]))

# Cross-tab with normalised row percentages
pd.crosstab(df['default'], df['collateral_type'], normalize='index').mul(100).round(1)

# Select only numeric columns and describe them together
df.select_dtypes(include='number').describe().T.sort_values('std', ascending=False)

# Binary encode all object columns at once
df[df.select_dtypes('object').columns] = df.select_dtypes('object').apply(lambda col: col.astype('category').cat.codes)